# Degrau de Potencial — Controle Difusional Reversível

Considera-se uma transferência de elétrons reversível entre as espécies $O$ e $R$:

$$O + ne^- \rightleftharpoons R$$

Assumindo eletrodo planar e coeficientes de difusão iguais para ambas as espécies, o transporte é descrito pela segunda lei de Fick para cada espécie:

$$\frac{\partial C_O}{\partial \tau} = D \frac{\partial^2 C_O}{\partial y^2} \tag{1.0a}$$

$$\frac{\partial C_R}{\partial \tau} = D \frac{\partial^2 C_R}{\partial y^2} \tag{1.0b}$$

As condições iniciais assumem $R$ inicialmente ausente:

$$C_O(y, 0) = C_O^* \tag{1.1a}$$

$$C_R(y, 0) = 0 \tag{1.1b}$$

A difusão semi-infinita é aplicada no seio da solução:

$$\lim_{y \to \infty} C_O(y, \tau) = C_O^* \tag{1.2a}$$

$$\lim_{y \to \infty} C_R(y, \tau) = 0 \tag{1.2b}$$

O equilíbrio de Nernst é imposto na interface eletrodo/solução, com $\theta = \exp\!\left(\frac{F}{RT}(E - E^0)\right)$:

$$C_O(0, \tau) = \frac{\theta}{1+\theta} C_O^* \quad \text{para} \quad \tau > 0 \tag{1.3a}$$

$$C_R(0, \tau) = \frac{1}{1+\theta} C_O^* \quad \text{para} \quad \tau > 0 \tag{1.3b}$$

Introduzindo as variáveis adimensionais:

$$x = \frac{y}{\sqrt{D\tau_s}}, \quad t = \frac{\tau}{\tau_s}, \quad c_{O,R} = \frac{C_{O,R}}{C_O^*}, \quad \eta = E - E^0, \quad f = \frac{F}{RT}$$

as equações tornam-se, para $t \geq 0$ e $0 \leq x \leq 6$:

$$\frac{\partial c_O}{\partial t} = \frac{\partial^2 c_O}{\partial x^2} \tag{1.0a}$$

$$\frac{\partial c_R}{\partial t} = \frac{\partial^2 c_R}{\partial x^2} \tag{1.0b}$$

$$c_O(x, 0) = 1 \tag{1.1a}$$

$$c_R(x, 0) = 0 \tag{1.1b}$$

$$\lim_{x \to \infty} c_O(x, t) = 1 \tag{1.2a}$$

$$\lim_{x \to \infty} c_R(x, t) = 0 \tag{1.2b}$$

com $\theta = \exp(f\eta)$:

$$c_O(0, t) = \frac{\theta}{1+\theta} \tag{1.3a}$$

$$c_R(0, t) = \frac{1}{1+\theta} \tag{1.3b}$$

As soluções analíticas para os perfis de concentração são:

$$c_O(x,t) = 1 - \frac{1}{1+\theta}\,\text{erfc}\!\left(\frac{x}{2\sqrt{t}}\right) \tag{2.0a}$$

$$c_R(x,t) = \frac{1}{1+\theta}\,\text{erfc}\!\left(\frac{x}{2\sqrt{t}}\right) \tag{2.0b}$$

A densidade de corrente é proporcional ao fluxo difusional na superfície do eletrodo:

$$j(\tau) = nFD \left(\frac{\partial C_O}{\partial y}\right)_{y=0} = nFD\frac{C_O^*}{\sqrt{D\tau}} \left(\frac{\partial c_O}{\partial x}\right)_{x=0}$$

Definindo a corrente adimensional:

$$i(t) = \frac{j(\tau)\sqrt{D\tau_s}}{nFD C_O^*}$$

e derivando as expressões analíticas da concentração, obtém-se:

$$i(t) = \frac{1}{\sqrt{\pi t}(1+\theta)} \tag{3.0}$$

## Bibliotecas:

In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

## Definindo o Modelo:

In [ ]:
model = pybamm.BaseModel()

concentration_o = pybamm.Variable("Concentração de O", domain="electrolyte")
concentration_r = pybamm.Variable("Concentração de R", domain="electrolyte")

overpotential = pybamm.Parameter("Sobrepotencial Aplicado [V]")
faraday       = pybamm.Parameter("Constante de Faraday [C.mol-1]")
gas_constant  = pybamm.Parameter("Constante dos Gases [J.K-1.mol-1]")
temperature   = pybamm.Parameter("Temperatura [K]")

flux_o = -pybamm.grad(concentration_o)   # fluxo difusional de O
flux_r = -pybamm.grad(concentration_r)   # fluxo difusional de R

model.rhs = {
    concentration_o: -pybamm.div(flux_o),  # lei de Fick para O (eq. 1.0a)
    concentration_r: -pybamm.div(flux_r),  # lei de Fick para R (eq. 1.0b)
}

# condições iniciais (eqs. 1.1a e 1.1b)
model.initial_conditions = {
    concentration_o: pybamm.Scalar(1),  # c_O(x,0) = 1: O uniforme e adimensionalizado
    concentration_r: pybamm.Scalar(0),  # c_R(x,0) = 0: R inicialmente ausente
}

# condições de contorno — Dirichlet
# esquerda (x=0): interface eletrodo/solução — equilíbrio de Nernst
# direita  (x=6): seio da solução            — difusão semi-infinita
nernst_factor = faraday / (gas_constant * temperature)   # f = F/(RT)
theta         = pybamm.exp(nernst_factor * overpotential) # θ = exp(f·η)

model.boundary_conditions = {
    concentration_o: {
        "left":  (theta / (1 + theta), "Dirichlet"),  # eq. 1.3a: c_O(0,t) = θ/(1+θ)
        "right": (pybamm.Scalar(1),    "Dirichlet"),  # eq. 1.2a: c_O(∞,t) = 1
    },
    concentration_r: {
        "left":  (1 / (1 + theta),    "Dirichlet"),  # eq. 1.3b: c_R(0,t) = 1/(1+θ)
        "right": (pybamm.Scalar(0),    "Dirichlet"),  # eq. 1.2b: c_R(∞,t) = 0
    },
}

model.variables = {
    "Concentração de O": concentration_o,
    "Concentração de R": concentration_r,
    "Fluxo de O":        flux_o,
    "Fluxo de R":        flux_r,
}

param = pybamm.ParameterValues(
    {
        "Sobrepotencial Aplicado [V]":        "[input]",
        "Constante de Faraday [C.mol-1]":     96485.3,
        "Constante dos Gases [J.K-1.mol-1]":  8.31446,
        "Temperatura [K]":                    298.15,
    }
)


# ── Geometria e Malha ─────────────────────────────────────────────────────────

x_variable = pybamm.SpatialVariable(
    "x", domain=["electrolyte"], coord_sys="cartesian"
)

geometry = {
    "electrolyte": {x_variable: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}
}

submesh_types  = {
    "electrolyte": pybamm.MeshGenerator(
        pybamm.Exponential1DSubMesh,
        submesh_params={
            "side": "left",
            "stretch": 5,
        },
    )
}
variable_points = {x_variable: 400}
mesh            = pybamm.Mesh(geometry, submesh_types, variable_points)


# ── Discretização ─────────────────────────────────────────────────────────────

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
discretisation  = pybamm.Discretisation(mesh, spatial_methods)

param.process_model(model)
param.process_geometry(geometry)
discretisation.process_model(model)

## Funções Analíticas:

In [ ]:
NERNST_FACTOR = 38.9217  # f = F/(RT) a 298.15 K [V⁻¹]

def theta_nernst(eta):
    """Parâmetro de Nernst numérico: θ = exp(f·η)"""
    return np.exp(NERNST_FACTOR * eta)

def analytical_concentration_o(x, t, eta):
    """Concentração analítica de O: c_O(x,t) = 1 - [1/(1+θ)] · erfc(x / 2√t)"""
    th = theta_nernst(eta)
    return 1 - (1 / (1 + th)) * special.erfc(x / (2 * np.sqrt(t)))

def analytical_concentration_r(x, t, eta):
    """Concentração analítica de R: c_R(x,t) = [1/(1+θ)] · erfc(x / 2√t)"""
    th = theta_nernst(eta)
    return (1 / (1 + th)) * special.erfc(x / (2 * np.sqrt(t)))

def analytical_current(t, eta):
    """Corrente analítica adimensional: i(t) = −1 / [√(πt)·(1+θ)]
    Negativa por convenção catódica (redução).
    """
    th = theta_nernst(eta)
    return -1 / (np.sqrt(np.pi * t) * (1 + th))

## Gráfico Estático:

In [ ]:
DEFAULT_ETA = -0.1  # sobrepotencial padrão [V]

static_solver  = pybamm.ScipySolver()
time           = np.linspace(1e-5, 1, 1000)
solution       = static_solver.solve(
    model, time, inputs={"Sobrepotencial Aplicado [V]": DEFAULT_ETA}
)

concentration_o_solution = solution["Concentração de O"]
concentration_r_solution = solution["Concentração de R"]
flux_o_solution          = solution["Fluxo de O"]

x_plot = np.linspace(0, 6, 200)
times  = [1, 0.1, 0.01, 0.001]
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for t_i, color in zip(times, colors):
    # ── espécie O ─────────────────────────────────────────────────────────
    ax2.plot(
        x_plot, concentration_o_solution(t=t_i, x=x_plot),
        ".", color=color, markersize=8,
        label=f"O — t = {t_i}",
    )
    # analítico sem label — linha de referência, não entra na legenda
    ax2.plot(
        x_plot, analytical_concentration_o(x_plot, t_i, DEFAULT_ETA),
        "-", color=color, linewidth=2,
    )
    # ── espécie R ─────────────────────────────────────────────────────────
    ax2.plot(
        x_plot, concentration_r_solution(t=t_i, x=x_plot),
        "s", color=color, markersize=5,
        label=f"R — t = {t_i}",
    )
    # analítico sem label — linha de referência, não entra na legenda
    ax2.plot(
        x_plot, analytical_concentration_r(x_plot, t_i, DEFAULT_ETA),
        ":", color=color, linewidth=2,
    )

ax2.set_xlabel(r"$x$")
ax2.set_ylabel(r"$c$")
ax2.set_xlim([0, 0.9])
ax2.set_ylim([0, 1.05])
ax2.legend(fontsize=7, ncol=2)
ax2.set_title(f"Perfis de concentração (η = {DEFAULT_ETA} V)")

ax1.plot(solution.t, flux_o_solution(solution.t, x=0), "r.", label="Numérico")
ax1.plot(solution.t, analytical_current(solution.t, DEFAULT_ETA), "b-", label="Analítico")
ax1.set_xlabel(r"$t$")
ax1.set_ylabel(r"$i$")
ax1.set_xlim([0.01, 1])
ax1.set_ylim([-6, 0])
ax1.legend()

plt.tight_layout()
plt.show()

O gráfico da esquerda mostra a corrente em função do tempo. A corrente decai com $t^{-1/2}$, comportamento análogo ao modelo de Cottrell, porém modulado pelo fator $(1+\theta)^{-1}$: quanto mais negativo o sobrepotencial $\eta$, menor $\theta$ e, portanto, maior a magnitude da corrente. O gráfico interativo permite explorar como a variação de $\eta$ afeta simultaneamente os perfis de concentração e a resposta de corrente.

O gráfico da direita mostra os perfis de concentração de $O$ e $R$ em quatro instantes de tempo distintos, para um sobrepotencial fixo $\eta = -0.1$ V. A concentração superficial de $O$ é $\theta/(1+\theta)$, enquanto que a de $R$ é $1/(1+\theta)$, ambas determinadas pelo equilíbrio de Nernst que é proporcional ao sobrepotencial aplicado. Para $\eta < 0$, tem-se $\theta < 1$, favorecendo a redução, ou seja o consumo de $O$ e formação de $R$. Os pontos representam a solução numérica e as linhas a solução analítica (eqs. 2.0a e 2.0b). O acordo entre as duas confirma a validade da discretização. À medida que o tempo avança, a camada de difusão se aprofunda. 

## Gráfico Interativo:

In [ ]:
x_plot       = np.linspace(0, 6, 200)
time_int     = np.linspace(1e-5, 1, 1000)
interactive_solver = pybamm.ScipySolver()

output = Output()

def plot(eta_str):
    with output:
        clear_output(wait=True)

        # validação da entrada
        try:
            eta = float(eta_str)
        except ValueError:
            print("Por favor, insira um número válido.")
            return

        if eta >= 0:
            print("Por favor, insira um sobrepotencial negativo (η < 0).")
            return

        solution = interactive_solver.solve(
            model, time_int,
            inputs={"Sobrepotencial Aplicado [V]": eta}
        )

        concentration_o_solution = solution["Concentração de O"]
        concentration_r_solution = solution["Concentração de R"]
        flux_o_solution          = solution["Fluxo de O"]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

        times  = [1, 0.1, 0.01, 0.001]
        colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

        for t_i, color in zip(times, colors):
            # ── espécie O ─────────────────────────────────────────────────
            ax2.plot(
                x_plot, concentration_o_solution(t=t_i, x=x_plot),
                ".", color=color, markersize=8,
                label=f"O — t = {t_i}",
            )
            # analítico sem label — linha de referência, não entra na legenda
            ax2.plot(
                x_plot, analytical_concentration_o(x_plot, t_i, eta),
                "-", color=color, linewidth=2,
            )
            # ── espécie R ─────────────────────────────────────────────────
            ax2.plot(
                x_plot, concentration_r_solution(t=t_i, x=x_plot),
                "s", color=color, markersize=5,
                label=f"R — t = {t_i}",
            )
            # analítico sem label — linha de referência, não entra na legenda
            ax2.plot(
                x_plot, analytical_concentration_r(x_plot, t_i, eta),
                ":", color=color, linewidth=2,
            )

        ax2.set_xlabel(r"$x$")
        ax2.set_ylabel(r"$c$")
        ax2.set_xlim([0, 0.9])
        ax2.set_ylim([0, 1.05])
        ax2.legend(fontsize=7, ncol=2)
        ax2.set_title(f"Perfis de concentração (η = {eta} V)")

        ax1.plot(solution.t, flux_o_solution(solution.t, x=0), "r.", label="Numérico")
        ax1.plot(solution.t, analytical_current(solution.t, eta), "b-", label="Analítico")
        ax1.set_xlabel(r"$t$")
        ax1.set_ylabel(r"$i$")
        ax1.set_xlim([0.01, 1])
        ax1.set_ylim([-6, 0])
        ax1.legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)


# ── Widgets ───────────────────────────────────────────────────────────────────

field_eta = Text(
    value="-0.1",
    description=r"$\eta$ [V]  ($\eta$ < 0):",
    style={"description_width": "initial"},
)

button = Button(description="Recalcular", button_style="success")

def on_click(b):
    plot(field_eta.value)

button.on_click(on_click)

interface = VBox([HBox([field_eta, button]), output])
display(interface)

# exibe o gráfico inicial com os valores padrão
plot(field_eta.value)